# 02 · Calidad del OCR, limpieza, enmascaramiento y split

**Etapas 2 y 3.** No necesita GPU (puede correr en Colab CPU o en local). Entradas: `data/processed/ocr.parquet`
y `data/processed/muestra_meta.parquet` del notebook 01.

1. CER y WER del OCR contra la columna `text` (la **única** función de `text`: medir el OCR), por ocupación.
2. Proporción de **años de referencia** leídos correctamente.
3. Umbral de confianza elegido con datos.
4. Limpieza (guiones, PII) y **enmascaramiento** de fugas, con conteo de CVs afectados.
5. Split 70/15/15 estratificado por ocupación × banda, semilla fija → `data/splits/`.

In [ ]:
!pip install -q "jiwer==4.0.0"

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/leruizdelavegat-max/HW_04_202602.git"
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ = Path("/content/drive/MyDrive/HW_04_202602")
    if not (RAIZ / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(RAIZ)], check=True)
    else:
        subprocess.run(["git", "-C", str(RAIZ), "pull", "--ff-only"], check=False)
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

In [ ]:
import json
import re

import jiwer
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

from src import config, data, texto

meta = pd.read_parquet(config.PROCESADOS / "muestra_meta.parquet")
ocr = pd.read_parquet(config.PROCESADOS / "ocr.parquet")
run = json.loads((config.PROCESADOS / "run_config.json").read_text())
TOPE = run["tope"]
df = ocr.merge(meta[["id", "text", "first_name", "last_name", config.TARGET]], on="id", how="inner")
df["lineas"] = df["lineas_json"].map(json.loads)
print(f"{len(df)} CVs con OCR · tope = {TOPE}")

## 1 · CER, WER y años leídos

Normalización mínima antes de comparar: minúsculas y espacios colapsados (no queremos castigar saltos de
línea, que el OCR no puede reproducir igual que la plantilla). El CER es la distancia de Levenshtein
por carácter dividida entre el largo de la referencia (sesión 10, §6); el WER, lo mismo por palabra.

In [ ]:
ref = df["text"].map(texto.normalizar_para_cer)
hyp = df["texto_ocr"].map(texto.normalizar_para_cer)
df["CER"] = [jiwer.cer(r, h) for r, h in zip(ref, hyp)]
df["WER"] = [jiwer.wer(r, h) for r, h in zip(ref, hyp)]
df["recall_anios"] = [texto.recall_anios(r, h) for r, h in zip(df["text"], df["texto_ocr"])]

calidad = df.groupby("job").agg(n=("id", "size"), CER_medio=("CER", "mean"), CER_mediana=("CER", "median"),
                                WER_medio=("WER", "mean"), anios_leidos=("recall_anios", "mean"),
                                conf_media=("conf_media", "mean"))
calidad.loc["TOTAL"] = [len(df), df.CER.mean(), df.CER.median(), df.WER.mean(), df.recall_anios.mean(), df.conf_media.mean()]
calidad.round(4).to_csv(config.TABLAS / "ocr_calidad_por_ocupacion.csv")
calidad.round(4)

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(data=df, x="CER", y="job", ax=ejes[0], color="#2a78d6", fliersize=2); ejes[0].set_title("CER por ocupación")
sns.boxplot(data=df, x="WER", y="job", ax=ejes[1], color="#eb6834", fliersize=2); ejes[1].set_title("WER por ocupación"); ejes[1].set_ylabel("")
sns.histplot(df["recall_anios"], bins=20, ax=ejes[2], color="#0ca30c"); ejes[2].set_title("Años de referencia encontrados en el OCR")
fig.tight_layout(); fig.savefig(config.FIGURAS / "ocr_cer_wer.png", dpi=130, bbox_inches="tight"); plt.show()

**¿Dónde falla el OCR?** Los 10 CVs con mayor CER y un diff de los años que no aparecieron:

In [ ]:
peores = df.nlargest(10, "CER")[["id", "job", "CER", "WER", "recall_anios", "dos_columnas", "conf_media"]]
display(peores)
for r in df.nlargest(3, "CER").itertuples():
    faltan = sorted(set(texto.anios(r.text)) - set(texto.anios(r.texto_ocr)))
    print(f"── {r.id} ({r.job}) CER {r.CER:.2%} · años de la referencia que faltan en el OCR: {faltan}")
    print("   OCR:", r.texto_ocr[:500].replace("\n", " | "))

## 2 · Umbral de confianza

Descartar líneas de baja confianza quita basura, pero también puede borrar fechas. Se elige el umbral
**que minimiza el CER medio** (con el recall de años como control), en vez de fijarlo a ojo.

In [ ]:
umbrales = [0.0, 0.3, 0.5, 0.6, 0.7, 0.8, 0.9]
filas = []
for u in umbrales:
    h = [texto.normalizar_para_cer(" ".join(l["text"] for l in ls if l["conf"] >= u)) for ls in df["lineas"]]
    filas.append({"umbral": u, "CER": np.mean([jiwer.cer(r, x) for r, x in zip(ref, h)]),
                  "recall_anios": np.nanmean([texto.recall_anios(r, x) for r, x in zip(df["text"], h)]),
                  "pct_lineas_descartadas": np.mean([l["conf"] < u for ls in df["lineas"] for l in ls])})
tabla_umbral = pd.DataFrame(filas)
tabla_umbral.to_csv(config.TABLAS / "umbral_confianza.csv", index=False)
UMBRAL = float(tabla_umbral.loc[tabla_umbral["CER"].idxmin(), "umbral"])
print("Umbral elegido:", UMBRAL)
tabla_umbral.round(4)

## 3 · Leer 50 textos y detectar fugas

Antes de enmascarar, se cuentan las coincidencias de cada patrón en el texto OCR **crudo** y se muestran
ejemplos para verificar que el regex atrapa lo que debe (y nada más). Ajusta `src/texto.py` si al leer
`output/ocr_muestra_50.txt` aparece una forma que no está cubierta.

In [ ]:
fugas = {tipo: df["texto_ocr"].map(lambda t, p=p: p.findall(t)) for tipo, p in texto.RE_FUGAS.items()}
for tipo, s in fugas.items():
    afectados = (s.map(len) > 0).sum()
    ejemplos = [m for ms in s if ms for m in ms][:8]
    print(f"{tipo:<18} CVs afectados: {afectados:4d}  ·  ejemplos: {ejemplos}")

## 4 · Limpieza y enmascaramiento

Orden: (1) quitar líneas bajo el umbral, (2) unir palabras partidas con guion y colapsar espacios,
(3) borrar PII — correo, URL, teléfono, dirección/ciudad-estado y el **nombre** del propio CV
(los nombres se usan solo para borrarlos, nunca como feature), (4) reemplazar fugas por `[MASK]`.

Se guardan tres variantes para la Pregunta 3 (sensibilidad a la limpieza). **Todas** llevan el
enmascaramiento: ningún modelo ve la respuesta.

| variante | umbral | PII y guiones | máscara |
|---|---|---|---|
| `crudo` | 0 | no | sí |
| `limpio` | 0 | sí | sí |
| `final` (por defecto) | elegido | sí | sí |

In [ ]:
def procesar(fila, umbral, pii=True):
    nombres = (fila.first_name, fila.last_name) if pii else ()
    if pii:
        return texto.limpiar(fila.lineas, umbral, nombres)
    t, c = texto.enmascarar("\n".join(l["text"] for l in fila.lineas if l["conf"] >= umbral))
    return t, c

res_final = [procesar(f, UMBRAL) for f in df.itertuples()]
df["texto_final"] = [t for t, _ in res_final]
df["texto_limpio"] = [procesar(f, 0.0)[0] for f in df.itertuples()]
df["texto_crudo"] = [procesar(f, 0.0, pii=False)[0] for f in df.itertuples()]
# La referencia `text` recibe la MISMA limpieza: solo se usa para la Pregunta 2 (costo del ruido del OCR)
df["texto_ref"] = [texto.limpiar_texto_plano(f.text, (f.first_name, f.last_name))[0] for f in df.itertuples()]

conteos = pd.DataFrame([c for _, c in res_final])
mascaras = {"cvs_con_alguna_mascara": int((conteos.sum(axis=1) > 0).sum()),
            **{f"cvs_{k}": int((conteos[k] > 0).sum()) for k in conteos},
            "mascaras_totales": int(conteos.values.sum())}
(config.TABLAS / "enmascaramiento.json").write_text(json.dumps(mascaras, indent=2))
print(json.dumps(mascaras, indent=2))

In [ ]:
# Verificación: ¿queda algo de PII o alguna fuga? ¿Sobreviven las fechas?
quedan_email = df["texto_final"].str.contains(texto.RE_EMAIL).sum()
quedan_tel = df["texto_final"].str.contains(texto.RE_TEL).sum()
quedan_fugas = sum(df["texto_final"].str.contains(p).sum() for p in texto.RE_FUGAS.values())
anios_conservados = np.nanmean([texto.recall_anios(a, b) for a, b in zip(df["texto_ocr"], df["texto_final"])])
print(f"Correos restantes: {quedan_email} · teléfonos: {quedan_tel} · fugas restantes: {quedan_fugas}")
print(f"Años del OCR que siguen en el texto limpio: {anios_conservados:.1%}")
print("\nEjemplo limpio:\n", df["texto_final"].iloc[0][:1200])

## 5 · Target, bandas y split 70/15/15

Estratificado por **ocupación × banda** con la misma semilla. Los IDs se guardan en `data/splits/` y
todos los modelos los leen de ahí. El test no se toca hasta el notebook 05.

In [ ]:
df["y"] = data.recortar_target(df[config.TARGET], TOPE)
df["banda"] = df["y"].map(config.banda)
estrato = df["job"] + "|" + df["banda"].astype(str)
tr, temp = train_test_split(df["id"], test_size=0.30, stratify=estrato, random_state=config.SEED)
va, te = train_test_split(temp, test_size=0.50, stratify=estrato[temp.index], random_state=config.SEED)
df["split"] = np.where(df["id"].isin(tr), "train", np.where(df["id"].isin(va), "val", "test"))
for nombre in ("train", "val", "test"):
    df.loc[df.split == nombre, ["id"]].to_csv(config.SPLITS / f"{nombre}_ids.csv", index=False)
display(pd.crosstab(df["job"], df["split"], margins=True))
display(pd.crosstab(df["banda"].map(dict(enumerate(config.NOMBRES_BANDAS))), df["split"], normalize="columns").round(3))

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df["y"], bins=range(0, int(TOPE) + 2), ax=ejes[0], color="#2a78d6")
ejes[0].set_title(f"Años de experiencia (recortado a {TOPE}) · en el tope: {(df.y >= TOPE).mean():.1%}")
sns.boxplot(data=df, x="y", y="job", ax=ejes[1], color="#eb6834"); ejes[1].set_title("Por ocupación"); ejes[1].set_ylabel("")
fig.tight_layout(); fig.savefig(config.FIGURAS / "distribucion_target.png", dpi=130, bbox_inches="tight"); plt.show()

## 6 · Guardar el dataset de modelado

- `dataset.parquet`: id, ocupación, target, split, las variantes de texto y métricas de OCR. **Sin**
  columnas prohibidas.
- `auditoria.parquet`: `sex` y `race`, separados; solo los usa la comprobación de equidad (Pregunta 4).

In [ ]:
cols = ["id", "job", "y", "banda", "split", "texto_final", "texto_limpio", "texto_crudo", "texto_ref",
        "texto_ocr", "CER", "WER", "recall_anios", "conf_media", "dos_columnas"]
df[cols].to_parquet(config.PROCESADOS / "dataset.parquet", index=False)
meta[["id", "sex", "race"]].to_parquet(config.PROCESADOS / "auditoria.parquet", index=False)
run.update({"umbral_confianza": UMBRAL, **{f"calidad_{k}": float(v) for k, v in calidad.loc["TOTAL"].items()}})
(config.PROCESADOS / "run_config.json").write_text(json.dumps(run, indent=2))
print("Guardado.", df.shape)

In [ ]:
import shutil
if EN_COLAB:
    paquete = Path("/content/entregables_02")
    shutil.rmtree(paquete, ignore_errors=True)
    for rel in ["data/splits", "data/processed/dataset.parquet", "data/processed/auditoria.parquet",
                "data/processed/run_config.json", "output"]:
        o, d = RAIZ / rel, paquete / rel
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(o, d, dirs_exist_ok=True) if o.is_dir() else shutil.copy2(o, d)
    shutil.make_archive("/content/entregables_02", "zip", paquete)
    from google.colab import files
    files.download("/content/entregables_02.zip")